In [14]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import JackageNormalizer
import tensorflow as tf
import numpy as np
import gradio as gr

from JackageNormalizer import normalize_persian_text
from tensorflow import keras
from transformers import TFBertModel
from transformers import AutoTokenizer

In [2]:
model_name = "HooshvareLab/bert-fa-base-uncased"
model_path = "/Users/hossein/Desktop/Hos/CODE/NLP/SentimentAnalyis_model/saved_model_tf"

In [3]:
loaded_model = keras.models.load_model(
    model_path,
    custom_objects={"TFBertModel": TFBertModel}
)

print("Model loaded successfully")
loaded_model.summary()

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Model loaded successfully
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, None)]               0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, None)]               0         []                            
 )                                                                                                
                                                                                                  
 tf_bert_model (TFBertModel  TFBaseModelOutputWithPooli   1628413   ['input_ids[0][0]',           
 )                           ngAndCrossAttentions(last_   44         'attention_mask[0][0]']      
                             hidden_state=(None, None,              

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

texts = [
    "این غذا خیلی عالی بود و واقعا دوستش داشتم!",
    "کاملاً ناامید شدم، اصلاً خوب نبود."
]

normalized_texts = [normalize_persian_text(t) for t in texts]

encoder = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="np"
)

In [ ]:
preds = loaded_model.predict(
    {"input_ids": encoder["input_ids"].astype("int32"),
     "attention_mask": encoder["attention_mask"].astype("int32")},
    verbose=0
)

for t, p in zip(texts, preds.reshape(-1)):
    print(f"Text: {t}")
    print(f"Prediction score : {float(p):.4f}\n")

Text: این غذا خیلی عالی بود و واقعا دوستش داشتم!
Prediction score (posetive): 0.9863

Text: کاملاً ناامید شدم، اصلاً خوب نبود.
Prediction score (posetive): 0.0101



### Gradio WebApp.

In [16]:
def _to_prob(x):
    p = float(np.squeeze(x))
    if p < 0.0 or p > 1.0:
        p = 1.0 / (1.0 + np.exp(-p))
    return max(0.0, min(1.0, p))

def infer(text, threshold=0.5, max_length=128):
    text = (text or "").strip()
    if not text:
        return "—", 0.0

    enc = tokenizer(
        [text],
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="tf"
    )

    input_ids = tf.cast(enc["input_ids"], tf.int32)
    attention_mask = tf.cast(enc["attention_mask"], tf.int32)
    inputs = [input_ids, attention_mask]

    if hasattr(loaded_model, "pred"):
        raw = loaded_model.pred(inputs)
    else:
        raw = loaded_model.predict(inputs, verbose=0)

    prob_pos = _to_prob(raw)
    label = "Positive" if prob_pos >= float(threshold) else "Negative"
    return label, float(round(prob_pos, 4))

with gr.Blocks(title="Persian Sentiment") as demo:
    gr.Markdown("## Persian Sentiment")
    gr.Markdown(
        "Type a sentence in Persian. The model returns a label and the positive probability."
    )

    txt = gr.Textbox(
        label="Input text",
        placeholder="...این غذا خیلی خوشمزه بود",
        lines=3
    )

    btn = gr.Button("Predict")

    out_label = gr.Textbox(label="Predicted label", interactive=False)
    out_prob  = gr.Number(label="Positive probability", interactive=False)

    btn.click(fn=infer, inputs=txt, outputs=[out_label, out_prob])

    gr.Examples(
        examples=[
            "این غذا خیلی خوشمزه بود!",
            "اصلاً راضی نبودم، تجربه‌ی بدی بود.",
            "معمولی بود؛ نه خوب نه بد.",
            "ارسال سریع و کیفیت عالی. حتماً دوباره سفارش می‌دم.",
        ],
        inputs=txt
    )

demo.launch(debug=False, share=True)

* Running on local URL:  http://127.0.0.1:7861


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


* Running on public URL: https://3f7e1300e46089e44c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
